# DPO: учиться по парам предпочтений

**Direct Preference Optimization: Your Language Model is Secretly a Reward Model** · Rafailov et al. · 2023

[Оригинальная статья](https://arxiv.org/abs/2305.18290)

**Цель урока:** Связать sequence log-probabilities с DPO objective, проверить masking, reference и знак обновления.

## Перед началом

Causal LM, log_softmax, response masks; вводный урок предпочтений.

Около 40–60 минут с заданиями. CPU, PyTorch и Matplotlib; без скачивания весов и датасетов. Время — ориентир для чтения и работы, а не benchmark выполнения.

## Задача и идея

DPO превращает предпочтения chosen/rejected в loss без отдельного обучения reward model в этом objective. Нужны policy и фиксированная reference. Отношение их вероятностей задаёт implicit reward; chosen должен выигрывать относительно rejected именно по этому отношению.

Просто увеличить вероятность chosen недостаточно для проверки: важна разница с reference. Dataset предпочтений может быть шумным и смещённым. Ниже используем три искусственные пары и маленькую categorical policy, чтобы видеть каждое число.

## Граница воспроизведения

DPO objective и masking. Нет реальной language generation, preference corpus или reproduction human eval статьи.

Код написан для этого урока по математическим идеям статьи. Это не скопированный авторский репозиторий. Успех assertions подтверждает указанные свойства реализации, но не результаты статьи.

## Формула → тензоры

`margin = beta * [(logpi_ch - logpi_rej) - (logref_ch - logref_rej)]`; `loss = -logsigmoid(margin)`.

Для LM logpi — сумма log probabilities ответа с causal shift и token mask. Среднее по токенам вместо суммы меняет objective и length effects. Reference заморожена; beta масштабирует margin. Логарифмы считаем через log_softmax, а не log(softmax), чтобы избежать потери устойчивости.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Шаг 1

Проверяем вычисление sequence log-probabilities. Изменение masked token не должно менять сумму.

In [ ]:
def sequence_logp(logits, targets, mask):
    token_logp = logits.log_softmax(-1).gather(-1, targets[..., None]).squeeze(-1)
    return (token_logp * mask).sum(-1)
logits = torch.randn(2, 4, 7)
targets = torch.randint(0, 7, (2, 4)); mask = torch.tensor([[1, 1, 0, 0], [1, 1, 1, 0]])
changed = logits.clone(); changed[mask == 0] = torch.randn_like(changed[mask == 0]) * 10
assert torch.equal(sequence_logp(logits, targets, mask), sequence_logp(changed, targets, mask))
print('sequence log-probs:', sequence_logp(logits, targets, mask))

## Шаг 2

У reference и policy в начале одинаковые logits: margin=0 и loss=log(2), даже если raw probabilities неравны.

In [ ]:
policy = nn.Parameter(torch.tensor([[0.2, -0.3], [-0.2, 0.8], [0., 0.]]))
reference = policy.detach().clone().log_softmax(-1)
beta = 0.5
def dpo():
    logp = policy.log_softmax(-1)
    margins = beta * ((logp[:, 0] - logp[:, 1]) - (reference[:, 0] - reference[:, 1]))
    return -F.logsigmoid(margins).mean(), margins
loss, initial_margin = dpo()
assert torch.allclose(initial_margin, torch.zeros(3))
assert abs(loss.item() - math.log(2)) < 1e-6
loss.backward()
print('chosen/rejected gradients:', policy.grad)
assert (policy.grad[:, 0] < 0).all() and (policy.grad[:, 1] > 0).all()

## Шаг 3

Reference остаётся неизменной, margins растут. График показывает objective, а не полезность ответов.

In [ ]:
frozen_reference = reference.clone()
optimizer = torch.optim.SGD([policy], lr=0.5)
history = []
for step in range(40):
    optimizer.zero_grad(set_to_none=True)
    loss, margins = dpo(); loss.backward(); optimizer.step()
    history.append(loss.item())
assert torch.equal(reference, frozen_reference)
assert (dpo()[1] > initial_margin).all()
plt.plot(history, color='#d73535'); plt.ylabel('DPO loss'); plt.xlabel('step'); plt.show()
print('final relative margins:', dpo()[1].detach())

## Как читать результат

Loss=log(2) на старте — хороший regression check для policy=reference. Неверная token mask или обновление reference изменит смысл обучения, даже если loss всё ещё падает. В настоящей задаче предпочтения нужно оценивать на held-out prompts.

## Самостоятельные эксперименты

1. Пересчитывайте reference после каждого шага: что произойдёт с margin?
2. Поменяйте chosen/rejected местами.
3. Сравните сумму и среднее log-probs для ответов разной длины.

Сначала запишите прогноз, затем меняйте по одному фактору. Для честного сравнения сбрасывайте ядро и выполняйте все ячейки с начала. Сохраните копию для своих изменений.

## Проверка рассуждений — после эксперимента

Если reference постоянно совпадает с policy, относительный margin снова обнуляется. Swap меняет знак сигнала. Mean меняет нормировку по длине; это отдельный objective, а не безусловное исправление DPO.

## Что записать в лабораторный журнал

- Гипотеза и изменённый параметр.
- Seed, размер данных, число шагов и используемая precision.
- Значения проверок до и после, график и объяснение расхождения с прогнозом.
- Что этот эксперимент **не** позволяет заключить.

Связанный раздел доступен по кнопке «Открыть раздел студии» над уроком.